# Track2 Mission 1-2 Workbench

이 노트북은 **Track2 실습지 미션 1~2**를 직접 실행/검증하기 위한 실습 워크벤치입니다.

- Mission 1: 킥오프 검수 + 고정 키워드 5종 프로브 + 소스 인벤토리
- Mission 2: 크로스 소스 Entity-to-Document 매핑 초안 생성 + 자동 검증


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import csv
import json
import re
import subprocess
import sys

def find_track2_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if (base / "generated" / "manifests" / "content_manifest.csv").exists():
            return base
        if (base / "track2/data" / "generated" / "manifests" / "content_manifest.csv").exists():
            return base / "track2/data"
    raise FileNotFoundError("track2/data 루트를 찾을 수 없습니다.")

ROOT = find_track2_root()
MANIFEST_CSV = ROOT / "generated" / "manifests" / "content_manifest.csv"
VALIDATOR_SCRIPT = ROOT / "verify_entity_document_mapping.py"

print(f"ROOT: {ROOT}")
print(f"MANIFEST: {MANIFEST_CSV}")
print(f"VALIDATOR: {VALIDATOR_SCRIPT}")


## Mission 1-1. Track1 인계 패키지 체크

아래 값을 실습팀이 채운 뒤 `all(...)` 결과가 `True`인지 확인하세요.

In [ ]:
handoff_check = {
    "workspace_id": True,
    "ontology_id": True,
    "model_summary": True,
    "entity_table_mapping_5plus": True,
    "top3_issues_with_workaround": True,
    "validation_log": True,
}

print(json.dumps(handoff_check, ensure_ascii=False, indent=2))
print("handoff_pass:", all(handoff_check.values()))


## Mission 1-2. 고정 키워드 5종 프로브

In [ ]:
def load_manifest_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = []
        for row in reader:
            title = (row.get("title") or "").strip()
            source = (row.get("source") or "").strip()
            keywords_raw = (row.get("keywords") or "").strip()
            keywords = [k.strip() for k in keywords_raw.split(";") if k.strip()]
            searchable = " ".join([title, *keywords]).lower()
            rows.append({
                "id": (row.get("id") or "").strip(),
                "title": title,
                "source": source,
                "keywords": keywords,
                "searchable": searchable,
            })
    return rows

manifest_rows = load_manifest_rows(MANIFEST_CSV)
print(f"manifest_rows: {len(manifest_rows)}")

probe_plan = [
    ("캠페인", "SummerPush", {"SharePoint", "Outlook"}),
    ("캠페인", "VIPRetention", {"SharePoint", "Outlook"}),
    ("상품", "AeroPhone X", {"Teams", "Outlook", "SharePoint"}),
    ("상품", "SmartWatch Pro", {"Teams", "Outlook", "SharePoint"}),
    ("고객맥락", "Platinum", {"Outlook", "OneDrive", "SharePoint"}),
]

probe_results = []
for category, keyword, expected_sources in probe_plan:
    hits = [r for r in manifest_rows if keyword.lower() in r["searchable"]]
    hit_sources = {r["source"] for r in hits}
    success = len(hits) > 0 and len(hit_sources & expected_sources) > 0
    probe_results.append({
        "category": category,
        "keyword": keyword,
        "hit_count": len(hits),
        "hit_sources": sorted(hit_sources),
        "expected_sources": sorted(expected_sources),
        "success": success,
    })

for row in probe_results:
    print(
        f"[{row['category']}] {row['keyword']}: "
        f"hits={row['hit_count']} sources={row['hit_sources']} success={row['success']}"
    )

success_count = sum(1 for r in probe_results if r["success"])
failed = [r["keyword"] for r in probe_results if not r["success"]]
print(f"\nkeyword_probe_success={success_count}/5")
print("kickoff_gate_pass:", success_count >= 4)
print("failed_keywords:", failed if failed else "-")


In [ ]:
team_name = "Team-A"
ontology_id = "<ONTOLOGY_ID>"
success_count = sum(1 for r in probe_results if r["success"])
failed = [r["keyword"] for r in probe_results if not r["success"]]

kickoff_check = f"""[TRACK2_KICKOFF_CHECK]
team={team_name}
ontologyId={ontology_id}
keywordProbe={success_count}/5
failedKeywords={';'.join(failed) if failed else '-'}
immediateAction=표기 정규화 및 권한/범위 재확인
[/TRACK2_KICKOFF_CHECK]"""

print(kickoff_check)


## Mission 1-3. M365 소스 인벤토리

In [ ]:
source_rows = defaultdict(list)
for row in manifest_rows:
    source_rows[row["source"]].append(row)

for source in ["SharePoint", "Outlook", "Teams", "OneDrive"]:
    rows = source_rows.get(source, [])
    print(f"{source}: {len(rows)}")
    for sample in rows[:2]:
        print(f"  - {sample['id']} | {sample['title']}")
    if not rows:
        print("  - (no rows)")


## Mission 2. 크로스 소스 Entity-to-Document 매핑 초안 생성

아래 셀은 실습지 체크 조건을 만족하도록 후보 매핑을 자동 생성합니다.

In [ ]:
required_entities = {
    "캠페인": ["SummerPush", "BackToSchool", "VIPRetention", "FlashWeek"],
    "상품": ["AeroPhone X", "SmartWatch Pro", "UltraBook 15", "DailyTee Cotton", "ComfyChair Home"],
    "고객등급": ["Platinum"],
}

core_products = ["AeroPhone X", "SmartWatch Pro", "UltraBook 15"]

def find_hits(entity_value: str) -> list[dict[str, str]]:
    token = entity_value.lower()
    return [r for r in manifest_rows if token in r["searchable"]]

mapping_rows = []
for entity_type, entity_values in required_entities.items():
    for entity_value in entity_values:
        hits = find_hits(entity_value)
        take_n = 2 if (entity_type == "상품" and entity_value in core_products) else 1
        for hit in hits[:take_n]:
            mapping_rows.append({
                "엔터티 유형": entity_type,
                "엔터티 값": entity_value,
                "매칭 문서 제목": hit["title"],
                "소스": hit["source"],
                "문서 링크/ID": hit["id"],
                "매칭 상태": "정확",
                "비고": "",
            })

# 의도된 표기 불일치 사례(정규화) 1건 자동 포함
alias_hits = [r for r in manifest_rows if "aero phone x" in r["searchable"]]
if alias_hits:
    hit = alias_hits[0]
    mapping_rows.append({
        "엔터티 유형": "상품",
        "엔터티 값": "Aero Phone X",
        "매칭 문서 제목": hit["title"],
        "소스": hit["source"],
        "문서 링크/ID": hit["id"],
        "매칭 상태": "부분",
        "비고": "원문: Aero Phone X -> 정규화: AeroPhone X",
    })

print(f"generated_mapping_rows={len(mapping_rows)}")

count_by_entity = defaultdict(set)
core_count = defaultdict(int)
for row in mapping_rows:
    if row["매칭 상태"] == "실패":
        continue
    count_by_entity[(row["엔터티 유형"], row["엔터티 값"])].add(row["문서 링크/ID"])
    if row["엔터티 유형"] == "상품" and row["엔터티 값"] in core_products:
        core_count[row["엔터티 값"]] += 1

for entity_type, entity_values in required_entities.items():
    covered = [v for v in entity_values if (entity_type, v) in count_by_entity]
    print(f"{entity_type}: {len(covered)}/{len(entity_values)} covered -> {covered}")

for product in core_products:
    print(f"core_product[{product}]={core_count[product]}")


In [ ]:
workbench_dir = ROOT / "generated" / "workbench"
workbench_dir.mkdir(parents=True, exist_ok=True)
mapping_csv = workbench_dir / "mission2_mapping_result_template.csv"
fieldnames = [
    "엔터티 유형",
    "엔터티 값",
    "매칭 문서 제목",
    "소스",
    "문서 링크/ID",
    "매칭 상태",
    "비고",
]

with mapping_csv.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f"saved: {mapping_csv}")


## Mission 2 자동 판정 실행

아래 셀은 `verify_entity_document_mapping.py`를 호출해 PASS/FAIL을 판정합니다.

In [ ]:
if not VALIDATOR_SCRIPT.exists():
    raise FileNotFoundError(f"validator not found: {VALIDATOR_SCRIPT}")

cmd = [
    sys.executable,
    str(VALIDATOR_SCRIPT),
    "--mapping-csv", str(mapping_csv),
    "--manifest-csv", str(MANIFEST_CSV),
    "--require-manifest-match",
    "--json-output", str(workbench_dir / "mission2_mapping_validation_report.json"),
]
print("run:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"validator failed with exit={result.returncode}")
